In [63]:
import numpy as np
import pandas as pd

cust = pd.read_csv("/data/cust_branch_dataset CLEANED.csv")

# ── Reproducibility ──────────────────────────────────────────────────────────
N_CUSTOMERS = 5000
rng = np.random.default_rng(123)

# ── Helpers ───────────────────────────────────────────────────────────────────
def lognorm_dollars(mean_log, sd_log):
    return float(np.round(np.exp(rng.normal(mean_log, sd_log)), 2))

def tier_from_wealth(w):
    if w < 0.55: return 0
    if w < 0.80: return 1
    if w < 0.95: return 2
    return 3

# ── Multipliers ───────────────────────────────────────────────────────────────
IBB_MULT   = {0: 0.70, 1: 1.00, 2: 1.80, 3: 3.60}
NIDDA_MULT = {0: 0.85, 1: 1.00, 2: 1.35, 3: 1.80}

# GTA-realistic loan caps per product type
LOAN_CAP_HOME = 900_000   # GTA avg mortgage ~$500K–$700K, ceiling $900K
LOAN_CAP_AUTO =  70_000   # Typical GTA vehicle financing ceiling
LOAN_CAP_PL   =  45_000   # Unsecured personal loan ceiling

# IBB deposit caps — prevent lognormal tail outliers at HNW tier + max lift
IBB_CAP_SAV =   500_000   # Savings ceiling
IBB_CAP_MM  =   750_000   # Money market ceiling
IBB_CAP_CDS =   500_000   # CD short term ceiling
IBB_CAP_CDL = 1_000_000   # CD long term ceiling

# ══════════════════════════════════════════════════════════════════════════════
# COMPOSITE LIFT FUNCTIONS (IBB deposits only)
# ══════════════════════════════════════════════════════════════════════════════

def get_coverage_lift(branches: int) -> float:
    """More nearby branches → deeper advisor access → higher IBB balances."""
    if branches == 0:  return 0.78   # -22%  No Nearby Branch
    if branches <= 2:  return 0.88   # -12%  Low Coverage (1-2)
    if branches <= 5:  return 1.00   #  base Moderate Coverage (3-5)
    if branches <= 10: return 1.14   # +14%  High Coverage (6-10)
    return 1.28                       # +28%  Very High Coverage (10+)

def get_tenure_lift(tenure_days: float) -> float:
    """Longer relationships → deeper product penetration → higher balances."""
    if tenure_days < 365:   return 0.88   # < 1 year
    if tenure_days < 1095:  return 0.95   # 1–3 years
    if tenure_days < 2555:  return 1.00   # 3–7 years (baseline)
    if tenure_days < 5110:  return 1.08   # 7–14 years
    return 1.15                            # 14+ years

def get_distance_lift(km: float) -> float:
    """Closer to primary branch → more advisor visits → higher balances."""
    if km < 0.5:  return 1.08   # very close      (<0.5 km)
    if km < 1.0:  return 1.04   # close           (0.5–1 km)
    if km < 2.0:  return 1.00   # baseline        (1–2 km)
    if km < 5.0:  return 0.96   # far             (2–5 km)
    return 0.91                  # very far        (5 km+)

def get_digital_lift(uses_mobile: int) -> float:
    """Mobile banking users are more engaged overall → higher balance growth."""
    return 1.05 if uses_mobile == 1 else 1.00

# Toronto neighbourhood wealth index.
# Based on well-documented socioeconomic stratification across the GTA.
# Source: StatsCan census income data by neighbourhood (2021).
NEIGHBOURHOOD_LIFT = {
    # Affluent core
    "Bridle Path":          1.30,
    "Rosedale":             1.28,
    "Forest Hill":          1.25,
    "Yorkville":            1.22,
    "Lawrence Park":        1.20,
    "Moore Park":           1.18,
    # Upper-middle
    "Leaside":              1.12,
    "Sunnybrook":           1.10,
    "Bedford Park":         1.10,
    "Summerhill":           1.08,
    "Davisville Village":   1.06,
    "Bloor West Village":   1.05,
    # Middle
    "Annex":                1.02,
    "Midtown":              1.00,
    "Leslieville":          0.98,
    "Riverdale":            0.97,
    "East York":            0.96,
    # Below average
    "Etobicoke North":      0.90,
    "Scarborough":          0.88,
    "Weston":               0.87,
    "Malvern":              0.85,
    "Rexdale":              0.86,
    "Jane and Finch":       0.82,
}

def get_neighbourhood_lift(neighbourhood: str) -> float:
    return NEIGHBOURHOOD_LIFT.get(neighbourhood, 1.00)

# ── Lookup Maps from cust dataframe ──────────────────────────────────────────
try:
    coverage_map      = cust.set_index("cust_id")["branches_within_1km"].to_dict()
    tenure_map        = cust.set_index("cust_id")["tenure_days"].to_dict()
    distance_map      = cust.set_index("cust_id")["primary_branch_km"].to_dict()
    digital_map       = cust.set_index("cust_id")["uses_mobile_banking"].to_dict()
    neighbourhood_map = cust.set_index("cust_id")["cust_neighbourhood"].to_dict()
    print("All lookup maps loaded from `cust` dataframe.")
except NameError:
    coverage_map = tenure_map = distance_map = digital_map = neighbourhood_map = {}
    print("WARNING: `cust` dataframe not found. All composite lifts defaulting to 1.0.")

# ── Main Generation Loop ──────────────────────────────────────────────────────
rows = []

for cid in range(1, N_CUSTOMERS + 1):

    # ── Composite IBB lift ────────────────────────────────────────────────────
    branches      = coverage_map.get(cid, 5)       # default = moderate coverage
    tenure_days   = tenure_map.get(cid, 1095)      # default = 3 years
    km            = distance_map.get(cid, 2.0)     # default = baseline (1–2 km)
    uses_mobile   = digital_map.get(cid, 0)        # default = non-digital
    neighbourhood = neighbourhood_map.get(cid, "") # default = baseline (1.0)

    composite_lift = (
        get_coverage_lift(branches)          *
        get_tenure_lift(tenure_days)         *
        get_distance_lift(km)                *
        get_digital_lift(uses_mobile)        *
        get_neighbourhood_lift(neighbourhood)
    )
    lift = float(np.clip(composite_lift, 0.65, 2.20))

    # Wealth proxy drives product mix and balance tiers
    wealth_proxy = rng.random()
    coverage_wealth_nudge = (
        -0.10 if branches == 0  else   # No Nearby Branch  → lower wealth
        -0.05 if branches <= 2  else   # Low Coverage      → slightly lower
         0.00 if branches <= 5  else   # Moderate          → baseline
         0.05 if branches <= 10 else   # High Coverage     → slightly higher
         0.10                          # Very High         → higher wealth
    )
    wealth_proxy = float(np.clip(wealth_proxy + coverage_wealth_nudge, 0.0, 1.0))
    tier         = tier_from_wealth(wealth_proxy)

    # ── DEPOSITS ──────────────────────────────────────────────────────────────
    deposit_types = {"Checking"}

    p_sav  = np.clip(0.20 + 0.35 * wealth_proxy, 0.20, 0.55)
    p_mm   = np.clip(0.03 + 0.15 * wealth_proxy, 0.03, 0.18)
    p_cd_s = np.clip(0.02 + 0.12 * wealth_proxy, 0.02, 0.14)
    p_cd_l = np.clip(0.01 + 0.07 * wealth_proxy, 0.01, 0.08)

    if rng.random() < p_sav:  deposit_types.add("Savings")
    if rng.random() < p_mm:   deposit_types.add("Money Market")
    if rng.random() < p_cd_s: deposit_types.add("CD Short Term")
    if rng.random() < p_cd_l: deposit_types.add("CD Long Term")

    for pt in deposit_types:
        if pt == "Checking":
            partial_lift = 1.0 + (lift - 1.0) * 0.40
            bal     = lognorm_dollars(8.70, 0.80) * NIDDA_MULT[tier] * partial_lift
            dep_cat = "NIDDA"
        elif pt == "Savings":
            bal     = min(lognorm_dollars(9.50, 0.95) * IBB_MULT[tier] * lift, IBB_CAP_SAV)
            dep_cat = "IBB"
        elif pt == "Money Market":
            bal     = min(lognorm_dollars(10.10, 1.00) * IBB_MULT[tier] * lift, IBB_CAP_MM)
            dep_cat = "IBB"
        elif pt == "CD Short Term":
            bal     = min(lognorm_dollars(10.50, 1.05) * IBB_MULT[tier] * lift, IBB_CAP_CDS)
            dep_cat = "IBB"
        else:  # CD Long Term
            bal     = min(lognorm_dollars(11.10, 1.15) * IBB_MULT[tier] * lift, IBB_CAP_CDL)
            dep_cat = "IBB"

        rows.append({
            "cust_id":          cid,
            "product_group":    "Deposit",
            "deposit_category": dep_cat,
            "product_type":     pt,
            "balance":          float(np.round(bal, 2)),
            "active_flag":      "Y"
        })

    # ── CREDIT ────────────────────────────────────────────────────────────────
    p_cc   = np.clip(0.55 + 0.21 * wealth_proxy, 0.55, 0.76)
    has_cc = rng.random() < p_cc
    cc_bal = lognorm_dollars(8.10, 0.90) if has_cc else 0.0

    rows.append({
        "cust_id":          cid,
        "product_group":    "Credit",
        "deposit_category": "N/A",
        "product_type":     "Credit Card",
        "balance":          float(np.round(cc_bal, 2)),
        "active_flag":      "Y" if has_cc else "N"
    })

    # ── LOANS ─────────────────────────────────────────────────────────────────
    p_home = np.clip(0.15 + 0.20 * wealth_proxy,       0.15, 0.35)
    p_auto = np.clip(0.18 + 0.07 * (1 - wealth_proxy), 0.18, 0.25)
    p_pl   = np.clip(0.08 + 0.07 * (1 - wealth_proxy), 0.08, 0.15)

    has_home = rng.random() < p_home
    has_auto = rng.random() < p_auto
    has_pl   = rng.random() < p_pl

    home_bal = min(lognorm_dollars(14.0, 0.45) if has_home else 0.0, LOAN_CAP_HOME)
    auto_bal = min(lognorm_dollars(10.1, 0.55) if has_auto else 0.0, LOAN_CAP_AUTO)
    pl_bal   = min(lognorm_dollars(9.4,  0.70) if has_pl   else 0.0, LOAN_CAP_PL)

    for loan_type, loan_bal, loan_flag in [
        ("Home Loan",     home_bal, has_home),
        ("Auto Loan",     auto_bal, has_auto),
        ("Personal Loan", pl_bal,   has_pl),
    ]:
        rows.append({
            "cust_id":          cid,
            "product_group":    "Loan",
            "deposit_category": "N/A",
            "product_type":     loan_type,
            "balance":          float(np.round(loan_bal, 2)),
            "active_flag":      "Y" if loan_flag else "N"
        })

# ── Build DataFrame ───────────────────────────────────────────────────────────
fact_cust_product = pd.DataFrame(rows)

# ── Credit card flag on full fact table (one row per cust_id, no duplicates) ──
cc_flag = (
    fact_cust_product
    .query("product_type == 'Credit Card'")
    [["cust_id", "active_flag"]]
    .drop_duplicates(subset=["cust_id"])
    .rename(columns={"active_flag": "has_credit_card"})
    .assign(has_credit_card=lambda df: df["has_credit_card"].map({"Y": 1, "N": 0}))
)
fact_cust_product = fact_cust_product.merge(cc_flag, on="cust_id", how="left")
fact_cust_product["has_credit_card"] = fact_cust_product["has_credit_card"].fillna(0).astype(int)

# ── Portfolio Balancing ───────────────────────────────────────────────────────
TARGET_COMBINED = 0.70
TARGET_CC_SHARE = 0.08

active    = fact_cust_product.query("active_flag == 'Y'").copy()
dep_total = active.query("product_group == 'Deposit'")["balance"].sum()

loan_mask = (
    (fact_cust_product["active_flag"]   == "Y") &
    (fact_cust_product["product_group"] == "Loan")
)
loan_total = fact_cust_product.loc[loan_mask, "balance"].sum()
if loan_total > 0:
    loan_scale = (dep_total * (TARGET_COMBINED - TARGET_CC_SHARE)) / loan_total
    fact_cust_product.loc[loan_mask, "balance"] = (
        fact_cust_product.loc[loan_mask, "balance"] * loan_scale
    ).clip(upper=LOAN_CAP_HOME).round(2)

cc_mask = (
    (fact_cust_product["active_flag"]   == "Y") &
    (fact_cust_product["product_group"] == "Credit")
)
cc_total = fact_cust_product.loc[cc_mask, "balance"].sum()
if cc_total > 0:
    cc_scale = (dep_total * TARGET_CC_SHARE) / cc_total
    fact_cust_product.loc[cc_mask, "balance"] = (
        fact_cust_product.loc[cc_mask, "balance"] * cc_scale
    ).round(2)

# ── Active-only view ──────────────────────────────────────────────────────────
fact_cust_product_active = fact_cust_product.query("active_flag == 'Y'").copy()
fact_cust_product_active = fact_cust_product_active.drop(columns=["active_flag"])



# ══════════════════════════════════════════════════════════════════════════════
# SANITY CHECKS
# ══════════════════════════════════════════════════════════════════════════════
dep_total2  = fact_cust_product_active.query("product_group=='Deposit'")["balance"].sum()
loan_total2 = fact_cust_product_active.query("product_group=='Loan'")["balance"].sum()
cc_total2   = fact_cust_product_active.query("product_group=='Credit'")["balance"].sum()
book_total  = dep_total2 + loan_total2 + cc_total2

print(f"Deposits total : ${dep_total2:,.0f}")
print(f"Loans total    : ${loan_total2:,.0f}")
print(f"Credit total   : ${cc_total2:,.0f}")
print(f"Total book     : ${book_total:,.0f}")
print(f"Loan/Deposit   : {loan_total2 / dep_total2:.3f}")
print(f"Credit/Deposit : {cc_total2 / dep_total2:.3f}")
print(f"Combined/Dep   : {(loan_total2 + cc_total2) / dep_total2:.3f}")
print(f"Max home loan  : ${fact_cust_product_active.query('product_type==\"Home Loan\"')['balance'].max():,.0f}")
print(f"Max auto loan  : ${fact_cust_product_active.query('product_type==\"Auto Loan\"')['balance'].max():,.0f}")
print(f"Max personal   : ${fact_cust_product_active.query('product_type==\"Personal Loan\"')['balance'].max():,.0f}")
print(f"Max CD Long    : ${fact_cust_product_active.query('product_type==\"CD Long Term\"')['balance'].max():,.0f}")
print(f"Max Savings    : ${fact_cust_product_active.query('product_type==\"Savings\"')['balance'].max():,.0f}")

# ── Product count distribution ────────────────────────────────────────────────
prod_counts = (
    fact_cust_product_active
    .groupby("cust_id")["product_type"]
    .nunique()
    .value_counts()
    .sort_index()
)
total_clients = prod_counts.sum()
print("\n── Product count distribution (active products only) ────────")
for n_prod, count in prod_counts.items():
    bar = "█" * int(count / total_clients * 40)
    print(f"  {n_prod} products : {count:5d} clients ({count/total_clients*100:4.1f}%)  {bar}")

# ── Lift verification by dimension ────────────────────────────────────────────
if coverage_map:

    def label_coverage(b):
        if b == 0:   return "1. No Nearby Branch"
        if b <= 2:   return "2. Low Coverage (1-2)"
        if b <= 5:   return "3. Moderate Coverage (3-5)"
        if b <= 10:  return "4. High Coverage (6-10)"
        return "5. Very High Coverage (10+)"

    def label_tenure(t):
        if t < 365:   return "1. <1 Year"
        if t < 1095:  return "2. 1-3 Years"
        if t < 2555:  return "3. 3-7 Years"
        if t < 5110:  return "4. 7-14 Years"
        return "5. 14+ Years"

    def label_distance(k):
        if k < 0.5:  return "1. Very Close (<0.5 km)"
        if k < 1.0:  return "2. Close (0.5–1 km)"
        if k < 2.0:  return "3. Moderate (1–2 km)"
        if k < 5.0:  return "4. Far (2–5 km)"
        return "5. Very Far (5 km+)"

    fca = fact_cust_product_active.copy()
    fca["coverage_tier"] = fca["cust_id"].map(coverage_map).map(label_coverage)
    fca["tenure_band"]   = fca["cust_id"].map(tenure_map).map(label_tenure)
    fca["distance_tier"] = fca["cust_id"].map(distance_map).map(label_distance)
    fca["digital_flag"]  = fca["cust_id"].map(digital_map).map(
                               lambda x: "Digital" if x == 1 else "Branch")

    dep_only = fca.query("product_group == 'Deposit'")

    def avg_dep_by(col):
        return (
            dep_only.groupby(["cust_id", col])["balance"].sum()
            .reset_index()
            .groupby(col)["balance"].mean()
            .round(0)
            .sort_index()
        )

    print("\n── Avg deposit per client by coverage tier ──────────────────")
    print(avg_dep_by("coverage_tier").to_string())

    print("\n── Avg deposit per client by tenure band ────────────────────")
    print(avg_dep_by("tenure_band").to_string())

    print("\n── Avg deposit per client by distance tier ──────────────────")
    print(avg_dep_by("distance_tier").to_string())

    print("\n── Avg deposit per client by digital flag ───────────────────")
    print(avg_dep_by("digital_flag").to_string())

fact_cust_product_active.head(10)

All lookup maps loaded from `cust` dataframe.
Deposits total : $151,509,584
Loans total    : $93,935,945
Credit total   : $12,120,767
Total book     : $257,566,295
Loan/Deposit   : 0.620
Credit/Deposit : 0.080
Combined/Dep   : 0.700
Max home loan  : $81,373
Max auto loan  : $6,329
Max personal   : $4,069
Max CD Long    : $1,000,000
Max Savings    : $500,000

── Product count distribution (active products only) ────────
  1 products :   471 clients ( 9.4%)  ███
  2 products :  1616 clients (32.3%)  ████████████
  3 products :  1733 clients (34.7%)  █████████████
  4 products :   874 clients (17.5%)  ██████
  5 products :   257 clients ( 5.1%)  ██
  6 products :    42 clients ( 0.8%)  
  7 products :     6 clients ( 0.1%)  
  8 products :     1 clients ( 0.0%)  

── Avg deposit per client by coverage tier ──────────────────
coverage_tier
1. No Nearby Branch            21419.0
2. Low Coverage (1-2)          29141.0
3. Moderate Coverage (3-5)     39743.0
4. High Coverage (6-10)        4402

,cust_id,product_group,deposit_category,product_type,balance,has_credit_card
0,1,Deposit,NIDDA,Checking,9376.33,1
1,1,Deposit,IBB,Savings,7013.11,1
2,1,Credit,N/A,Credit Card,1846.43,1
6,2,Deposit,NIDDA,Checking,5438.26,1
7,2,Deposit,IBB,Savings,24042.58,1
8,2,Credit,N/A,Credit Card,336.71,1
9,2,Loan,N/A,Home Loan,81372.60,1
12,3,Deposit,NIDDA,Checking,3960.82,0
17,4,Deposit,NIDDA,Checking,8332.07,1
18,4,Credit,N/A,Credit Card,8891.83,1


In [64]:
# Export active-only
fact_cust_product.to_csv("/data/fact_cust_product.csv", index=False)